# PathoCellBench Cell Type Prediction and Evaluation

This notebook performs cell type prediction on the PathoCellBench dataset using CellWhisperer.
It uses the get_performance_metrics_left_vs_right function for evaluation.

In [ ]:
import torch
import numpy as np
import pandas as pd
import anndata
import json
import logging
from pathlib import Path
from typing import List

# Import CellWhisperer evaluation functions
from cellwhisperer.validation.zero_shot.functions import (
    get_performance_metrics_left_vs_right,
)
from cellwhisperer.utils.model_io import load_cellwhisperer_model

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
# Get paths from snakemake
model_path = snakemake.input.model
adata_path = snakemake.input.adata
image_path = snakemake.input.image

output_results = Path(snakemake.output.results)
output_per_class = Path(snakemake.output.per_class_metrics)
output_confusion = Path(snakemake.output.confusion_matrix)

seed = int(snakemake.wildcards.seed)

logger.info(f"Model: {model_path}")
logger.info(f"Data: {adata_path}")
logger.info(f"Seed: {seed}")

In [ ]:
# Set random seed for reproducibility
torch.manual_seed(seed)
np.random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

In [ ]:
# Load the model
logger.info("Loading CellWhisperer model...")
(
    pl_model,
    text_processor,
    transcriptome_processor,
    image_processor,
) = load_cellwhisperer_model(model_path=model_path, eval=True)
model = pl_model.model
logger.info("Model loaded successfully")

In [ ]:
# Load the PathoCellBench data
logger.info(f"Loading AnnData from {adata_path}")
adata = anndata.read_h5ad(adata_path)
logger.info(f"Loaded {adata.n_obs} cells with {adata.n_vars} genes")

# Get cell types
cell_types = adata.obs['cell_type'].values
unique_cell_types = sorted(list(set(cell_types)))
logger.info(f"Found {len(unique_cell_types)} unique cell types: {unique_cell_types}")

# Print cell type distribution
cell_type_counts = pd.Series(cell_types).value_counts()
logger.info(f"Cell type distribution:\n{cell_type_counts}")

In [ ]:
# Prepare text descriptions for each cell type
# These will be used as the "right" side in the evaluation
cell_type_descriptions = [
    f"This is a histology image showing {cell_type} cells"
    for cell_type in unique_cell_types
]

logger.info(f"Created {len(cell_type_descriptions)} cell type descriptions")
for i, desc in enumerate(cell_type_descriptions[:3]):
    logger.info(f"  Example {i+1}: {desc}")

In [ ]:
# Create mapping from cell type to index
cell_type_to_idx = {ct: i for i, ct in enumerate(unique_cell_types)}

# Create correct indices for evaluation
# For each cell (left), the correct cell type description (right) is at this index
correct_right_idx_per_left = [cell_type_to_idx[ct] for ct in cell_types]

logger.info(f"Created {len(correct_right_idx_per_left)} correct index mappings")

In [ ]:
# Run evaluation using image data (set use_image_data=True)
logger.info("Running cell type prediction evaluation...")
logger.info("This evaluates how well image embeddings match text descriptions of cell types")

metrics, per_class_df = get_performance_metrics_left_vs_right(
    model=model,
    left_input=adata,  # Image data will be extracted from adata
    right_input=cell_type_descriptions,  # Text descriptions of cell types
    correct_right_idx_per_left=correct_right_idx_per_left,
    average_mode=None,  # No averaging - evaluate at single-cell level
    grouping_keys=None,
    batch_size=128,
    score_norm_method="zscore",  # Normalize scores using z-score
    report_per_class_metrics=True,
    right_as_classes=True,
    use_image_data=True,  # Use image data instead of transcriptome
)

logger.info("Evaluation completed successfully")

In [ ]:
# Extract and display key metrics
results = {
    'n_cells': int(adata.n_obs),
    'n_cell_types': len(unique_cell_types),
    'cell_types': unique_cell_types,
    'seed': seed,
}

# Add macro-averaged metrics
for metric_name, metric_value in metrics.items():
    if isinstance(metric_value, torch.Tensor):
        results[metric_name] = float(metric_value.item())
    else:
        results[metric_name] = float(metric_value)

# Display key results
logger.info("\n=== Cell Type Classification Results ===")
logger.info(f"Accuracy (macro): {results.get('accuracy_macroAvg', 0):.4f}")
logger.info(f"F1 score (macro): {results.get('f1_macroAvg', 0):.4f}")
logger.info(f"Precision (macro): {results.get('precision_macroAvg', 0):.4f}")
logger.info(f"AUROC (macro): {results.get('rocauc_macroAvg', 0):.4f}")
logger.info(f"Recall@1 (macro): {results.get('recall_at_1_macroAvg', 0):.4f}")
logger.info(f"Recall@5 (macro): {results.get('recall_at_5_macroAvg', 0):.4f}")

In [ ]:
# Display per-class metrics
logger.info("\n=== Per-Class Metrics ===")
logger.info(f"\n{per_class_df.to_string()}")

# Identify best and worst performing cell types
best_f1 = per_class_df.nlargest(3, 'f1')
worst_f1 = per_class_df.nsmallest(3, 'f1')

logger.info("\n=== Best Performing Cell Types (by F1) ===")
logger.info(f"\n{best_f1[['f1', 'precision', 'accuracy', 'n_samples_in_class']].to_string()}")

logger.info("\n=== Worst Performing Cell Types (by F1) ===")
logger.info(f"\n{worst_f1[['f1', 'precision', 'accuracy', 'n_samples_in_class']].to_string()}")

In [ ]:
# Save results
output_results.parent.mkdir(parents=True, exist_ok=True)

with open(output_results, 'w') as f:
    json.dump(results, f, indent=2)
logger.info(f"Saved results to {output_results}")

# Save per-class metrics
per_class_df.to_csv(output_per_class, index=True)
logger.info(f"Saved per-class metrics to {output_per_class}")

# Extract and save confusion matrix
confusion_cols = [col for col in per_class_df.columns if col.startswith('n_samples_predicted_as_')]
if confusion_cols:
    confusion_df = per_class_df[confusion_cols]
    # Rename columns to remove prefix
    confusion_df.columns = [col.replace('n_samples_predicted_as_', '') for col in confusion_df.columns]
    confusion_df.to_csv(output_confusion, index=True)
    logger.info(f"Saved confusion matrix to {output_confusion}")
else:
    logger.warning("No confusion matrix columns found in per_class_df")
    # Create an empty file
    pd.DataFrame().to_csv(output_confusion)

logger.info("Cell type prediction evaluation completed!")